In [1]:
import pandas as pd
import os

CLEAN  = "../data/clean"
MASTER = "../data/master"
os.makedirs(MASTER, exist_ok=True)

bikes   = pd.read_csv(f"{CLEAN}/bikes_clean.csv")
sales   = pd.read_csv(f"{CLEAN}/sales_clean.csv")
states  = pd.read_csv(f"{CLEAN}/ev_states_clean.csv")
reviews = pd.read_csv(f"{CLEAN}/reviews_clean.csv")

print("Loaded from clean:")
print(f"  bikes   → {bikes.shape}  | columns start: {bikes.columns[:4].tolist()}")
print(f"  sales   → {sales.shape}  | columns start: {sales.columns[:4].tolist()}")
print(f"  reviews → {reviews.shape}| columns start: {reviews.columns[:4].tolist()}")
print(f"  states  → {states.shape} | columns start: {states.columns[:4].tolist()}")

Loaded from clean:
  bikes   → (66, 21)  | columns start: ['bike_id', 'brand_id', 'name', 'brand']
  sales   → (3936, 10)  | columns start: ['bike_id', 'brand_id', 'name', 'year']
  reviews → (7827, 10)| columns start: ['bike_id', 'brand_id', 'bike_name', 'brand']
  states  → (1200, 7) | columns start: ['state', 'year', 'month', 'month_date']


In [2]:
# Summarize monthly → annual (keep bike_id and brand_id)
annual_sales = sales.groupby(["bike_id","brand_id","name","year"]).agg(
    annual_units = ("units_sold","sum")
).reset_index()

print(f"Annual sales → {annual_sales.shape}")
annual_sales.head(3)

Annual sales → (328, 5)


,bike_id,brand_id,name,year,annual_units
0,BK001,BR01,Hero Splendor Plus,2019,3876803
1,BK001,BR01,Hero Splendor Plus,2020,3044541
2,BK001,BR01,Hero Splendor Plus,2021,3650464


In [3]:
# Merge on bike_id — most reliable since names can have spacing issues
master = pd.merge(
    bikes,
    annual_sales,
    on=["bike_id","brand_id","name"],   # merge on all 3 for safety
    how="left"
)

print(f"After bikes + sales merge → {master.shape}")
master[["bike_id","brand_id","name","year","annual_units"]].head(5)

After bikes + sales merge → (328, 23)


,bike_id,brand_id,name,year,annual_units
0,BK001,BR01,Hero Splendor Plus,2019,3876803
1,BK001,BR01,Hero Splendor Plus,2020,3044541
2,BK001,BR01,Hero Splendor Plus,2021,3650464
3,BK001,BR01,Hero Splendor Plus,2022,4205779
4,BK001,BR01,Hero Splendor Plus,2023,4443445


In [4]:
avg_rating = reviews.groupby("bike_id").agg(
    avg_rating    = ("rating","mean"),
    total_reviews = ("rating","count")
).reset_index()

avg_rating["avg_rating"] = avg_rating["avg_rating"].round(2)

# Merge on bike_id — clean and reliable
master = pd.merge(master, avg_rating, on="bike_id", how="left")
master["total_reviews"] = master["total_reviews"].fillna(0).astype(int)

print(f"After adding ratings → {master.shape}")
master[["bike_id","brand_id","name","annual_units","avg_rating","total_reviews"]].head(5)

After adding ratings → (328, 25)


,bike_id,brand_id,name,annual_units,avg_rating,total_reviews
0,BK001,BR01,Hero Splendor Plus,3876803,4.17,68
1,BK001,BR01,Hero Splendor Plus,3044541,4.17,68
2,BK001,BR01,Hero Splendor Plus,3650464,4.17,68
3,BK001,BR01,Hero Splendor Plus,4205779,4.17,68
4,BK001,BR01,Hero Splendor Plus,4443445,4.17,68


In [5]:
print("=== MASTER DATASET VALIDATION ===\n")
print(f"Total rows      : {len(master)}")
print(f"Unique bikes    : {master['bike_id'].nunique()}")
print(f"EV bikes        : {master[master['is_ev']==True]['bike_id'].nunique()}")
print(f"Petrol bikes    : {master[master['is_ev']==False]['bike_id'].nunique()}")
print(f"Years covered   : {sorted(master['year'].dropna().unique().tolist())}")
print(f"\nNull check:")
print(master[["bike_id","brand_id","name","year","annual_units"]].isnull().sum())
print(f"\nTop 5 bikes by 2023 sales:")
top5 = (master[master["year"]==2023]
        .sort_values("annual_units", ascending=False)
        .head(5)[["bike_id","brand_id","name","segment","annual_units","price_inr"]])
print(top5.to_string(index=False))

=== MASTER DATASET VALIDATION ===

Total rows      : 328
Unique bikes    : 66
EV bikes        : 16
Petrol bikes    : 50
Years covered   : [2019, 2020, 2021, 2022, 2023]

Null check:
bike_id         0
brand_id        0
name            0
year            0
annual_units    0
dtype: int64

Top 5 bikes by 2023 sales:
bike_id brand_id               name  segment  annual_units  price_inr
  BK001     BR01 Hero Splendor Plus Commuter       4443445      72000
  BK010     BR02    Honda Activa 6G  Scooter       3931147      74000
  BK002     BR01     Hero HF Deluxe Commuter       2975935      58000
  BK031     BR04       Bajaj CT 110 Commuter       1505850      55000
  BK003     BR01   Hero Passion Pro Commuter       1495732      78000


In [6]:
# Save master — bike_id and brand_id are first columns automatically
master.to_csv(f"{MASTER}/master_bike_data.csv", index=False)
sales.to_csv(f"{MASTER}/sales_monthly.csv",     index=False)
states.to_csv(f"{MASTER}/ev_states.csv",        index=False)
reviews.to_csv(f"{MASTER}/reviews.csv",         index=False)

print("✅ All files saved to data/master/")
print(f"  master_bike_data.csv → {master.shape}")
print(f"  sales_monthly.csv    → {sales.shape}")
print(f"  ev_states.csv        → {states.shape}")
print(f"  reviews.csv          → {reviews.shape}")
print(f"\nFirst 2 columns of master: {master.columns[:4].tolist()}")

✅ All files saved to data/master/
  master_bike_data.csv → (328, 25)
  sales_monthly.csv    → (3936, 10)
  ev_states.csv        → (1200, 7)
  reviews.csv          → (7827, 10)

First 2 columns of master: ['bike_id', 'brand_id', 'name', 'brand']
